## Setup

In [1]:
from googleapiclient.discovery import build
import json
import re
from collections import Counter
from datetime import datetime
from googleapiclient.discovery import build
import pandas as pd
import numpy as np
import glob

In [2]:
with open ("../performative.txt", "r") as f:
    lines = f.readlines()
    API_KEY = lines[8].strip()
    
yt = build("youtube", "v3", developerKey=API_KEY)

### Static/Maps

In [3]:
GENRE_MAP = {
    "1":  "Film & Animation",
    "2":  "Autos & Vehicles",
    "10": "Music",
    "15": "Pets & Animals",
    "17": "Sports",
    "19": "Travel & Events",
    "20": "Gaming",
    "22": "People & Blogs",
    "23": "Comedy",
    "24": "Entertainment",
    "25": "News & Politics",
    "26": "Howto & Style",
    "27": "Education",
    "28": "Science & Technology",
    "29": "Nonprofits & Activism",
}

In [ ]:
QUERIES = [
    "video essay film analysis",
    "philosophy explained documentary",
    "internet culture deep dive",
    "history documentary essay channel",
    "game design analysis",
]
MAX_PER_QUERY = 50  # how many channels to collect per keyword/phrase


## Script

In [9]:
def search_channels(query, max_results, seen):
    collected = []
    next_page = None

    while len(collected) < max_results:
        resp = yt.search().list(
            part="snippet",
            q = query, 
            type="channel",
            relevanceLangauge = "en",
            maxResults=50,
            pageToken = next_page,
        ).execute()

        for item in resp["items"]:
            cid = item["snippet"]["channelId"]
            if cid not in seen:
                seen.add(cid)
                collected.append({
                    "channel_id":   item["snippet"]["channelId"],
                    "channel_name": item["snippet"]["title"],
                })
            if len(collected) >= max_results:
                break
        
        next_page = resp.get("nextPageToken")
        if not next_page:
            break
    return collected

## Running

In [ ]:
all_channels = []
seen = set()

for query in QUERIES:
    print(f"Searching: {query}")
    results = search_channels(query, MAX_PER_QUERY, seen)
    print(f" got {len(results)} channels")
    all_channels.extend(results)

with open("../data/processed/channels_missing.json", "r") as f:
    existing = json.load(f)
combined = existing + all_channels
with open("../data/processed/channels_missing.json", "w") as f:
    json.dump(all_channels, f, indent=2)